# Watermarking LLM Text: A Signal Hidden in the Sampling

The full code of [Watermarking LLM Text: A Signal Hidden in the Sampling](https://residual-stream.lvtan71.workers.dev/posts/watermarking-llm-text/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/watermarking-llm-text) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/watermarking-llm-text.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 35 min end to end on the RTX 3090 machine that saved the outputs below.
The texts are sampled, so detection counts vary by machine: at δ = 2, 7 of 10 were detected on an RTX 3090 against 5 of 10 in the post.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 35 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q numpy "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "watermarking-llm-text"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/watermarking-llm-text


In [2]:
!tar xzf blog-fb57e84.tar.gz

## `z_test.py`

The detector's z-test worked through for γ = 0.25, δ = 2: the green share of a flat distribution, z at 50 and 200 tokens, tokens needed for z = 4, and the false-positive rate at z > 4 from the normal tail and the exact binomial.

In [3]:
!{sys.executable} z_test.py

gamma = 0.25, delta = 2.0: a flat distribution's green share is 0.711
z for T = 200 tokens at that share: 15.1

green share  z at T=50  z at T=200  tokens for z = 4
      0.300        0.8         1.6              1201
      0.350        1.6         3.3               301
      0.400        2.4         4.9               134
      0.500        4.1         8.2                48
      0.711        7.5        15.1                15

false-positive rate at z > 4: normal tail 3.2e-05; exact at T = 50: 1.2e-04; exact at T = 200: 6.2e-05; exact at T = 1000: 4.8e-05


The post printed:

```text
gamma = 0.25, delta = 2.0: a flat distribution's green share is 0.711
z for T = 200 tokens at that share: 15.1

green share  z at T=50  z at T=200  tokens for z = 4
      0.300        0.8         1.6              1201
      0.350        1.6         3.3               301
      0.400        2.4         4.9               134
      0.500        4.1         8.2                48
      0.711        7.5        15.1                15

false-positive rate at z > 4: normal tail 3.2e-05; exact at T = 50: 1.2e-04; exact at T = 200: 6.2e-05; exact at T = 1000: 4.8e-05
```

## `wm.py`

The Kirchenbauer et al. watermark on Qwen2.5-1.5B-Instruct: 10 prompts, 200 tokens each, at δ = 0, 1, 2 and 4; green share, z, detections and quality (NLL). Then z against entropy at δ = 2, and two attacks: tokens replaced at random, and a paraphrase by the same model.

In [4]:
!{sys.executable} wm.py

gamma = 0.25, 200 tokens per text, temperature 0.7, 10 prompts
delta  green share   z (mean, min)  detected z>4  NLL (nats/token)  entropy


    0        0.254      0.1  -1.0         0/10             0.689     0.69


    1        0.321      2.3   0.3         1/10             0.779     0.73


    2        0.395      4.7   2.0         7/10             0.922     0.78


    4        0.538      9.4   4.9        10/10             1.348     0.79

delta = 2, texts from lowest to highest entropy:
  entropy  0.44  0.49  0.80  0.80  0.82  0.83  0.86  0.87  0.95  0.97
  z         4.1   3.4   5.1   6.5   6.0   2.0   5.9   3.9   5.9   4.4
  correlation 0.31

attacks on the delta = 2 texts: mean z (texts still detected at z > 4)


  replace 10% of tokens at random: z   3.7 (5/10)


  replace 20% of tokens at random: z   3.0 (2/10)


  replace 30% of tokens at random: z   2.1 (0/10)


  replace 50% of tokens at random: z   1.1 (0/10)


  paraphrase by the same model, unwatermarked: z   2.6 (2/10)


The post printed:

```text
gamma = 0.25, 200 tokens per text, temperature 0.7, 10 prompts
delta  green share   z (mean, min)  detected z>4  NLL (nats/token)  entropy
    0        0.247     -0.1  -1.5         0/10             0.713     0.69
    1        0.312      2.0  -0.3         1/10             0.768     0.70
    2        0.377      4.1   2.0         5/10             0.914     0.79
    4        0.550      9.8   4.9        10/10             1.410     0.82

delta = 2, texts from lowest to highest entropy:
  entropy  0.44  0.49  0.76  0.83  0.84  0.87  0.89  0.90  0.95  0.97
  z         4.1   3.4   3.8   2.0   5.4   3.9   5.7   2.9   5.9   4.4
  correlation 0.27

attacks on the delta = 2 texts: mean z (texts still detected at z > 4)
  replace 10% of tokens at random: z   3.4 (3/10)
  replace 20% of tokens at random: z   2.4 (1/10)
  replace 30% of tokens at random: z   2.0 (0/10)
  replace 50% of tokens at random: z   1.1 (0/10)
  paraphrase by the same model, unwatermarked: z   2.5 (2/10)
```

## `entropy_paraphrase.py`

The δ = 2 texts generated again with a log of every step: the green rate by the step's entropy, and how many token pairs each paraphrase shares with its original. Takes the model and functions from `wm.py`.

In [5]:
!{sys.executable} entropy_paraphrase.py

delta = 2: 2000 steps, green rate by the step's entropy (nats)
  entropy 0.0 to 0.1:   492 steps ( 25%), green 29%
  entropy 0.1 to 0.5:   365 steps ( 18%), green 30%
  entropy 0.5 to 1.0:   452 steps ( 23%), green 37%
  entropy 1.0 to 2.0:   553 steps ( 28%), green 52%
  entropy 2.0 to max:   138 steps (  7%), green 61%

paraphrases: share of their token pairs that also occur in the original


  text 0: 58% shared, z  2.3, starts 'Parks in urban areas are crucial'


  text 1: 99% shared, z  4.9, starts 'A morning at a fishing village is'


  text 2: 76% shared, z  2.4, starts 'A bicycle remains upright based on the'


  text 3: 100% shared, z  2.0, starts 'The sun was shining brightly, casting'


...

  text 5: 74% shared, z  3.1, starts 'The history of bread is a captivating'


  text 6: 30% shared, z  2.0, starts "From [Friend's Name],\n\nI"


  text 7: 30% shared, z  0.7, starts 'the sky appears blue during the day'


  text 8: 78% shared, z  4.2, starts 'Introducing our new wooden desk lamp'


  text 9: 66% shared, z  3.3, starts 'From a high vantage point,'


The post printed:

```text
delta = 2: 2000 steps, green rate by the step's entropy (nats)
  entropy 0.0 to 0.1:   482 steps ( 24%), green 28%
  entropy 0.1 to 0.5:   368 steps ( 18%), green 27%
  entropy 0.5 to 1.0:   462 steps ( 23%), green 37%
  entropy 1.0 to 2.0:   539 steps ( 27%), green 50%
  entropy 2.0 to max:   149 steps (  7%), green 55%

paraphrases: share of their token pairs that also occur in the original
  text 0: 78% shared, z  1.5, starts 'Parks have been a cornerstone of'
  text 1: 96% shared, z  2.4, starts 'A morning at a fishing village is'
  text 2: 76% shared, z  2.4, starts 'A bicycle remains upright based on the'
  text 3: 100% shared, z  2.0, starts 'The sun was shining brightly, casting'
  text 4: 62% shared, z  1.8, starts 'There are various qualities that make a'
  text 5: 74% shared, z  3.1, starts 'The history of bread is a captivating'
  text 6: 44% shared, z  2.1, starts "Dear [Friend's Name],\n\nI"
  text 7: 30% shared, z  0.7, starts 'the sky appears blue during the day'
  text 8: 78% shared, z  4.2, starts 'Introducing our new wooden desk lamp'
  text 9: 59% shared, z  4.6, starts 'From a high vantage point,'
```

## `human.py`

False positives on human text: every 200-token window of the blog's posts scored with `wm.py`'s key, counting every pair and each distinct pair once. Loads only the tokenizer.

In [6]:
!{sys.executable} human.py

1135 windows of 200 tokens from 52 posts; a standard normal has mean 0, std 1
every pair         z mean -0.20 (standard error 0.033), std 1.11, max 3.92; above 4: 0; above 2: 2.6% (normal: 2.3%)
unique pairs only  z mean -0.16 (standard error 0.029), std 0.99, max 3.03; above 4: 0; above 2: 1.5% (normal: 2.3%)


The post printed:

```text
1135 windows of 200 tokens from 52 posts; a standard normal has mean 0, std 1
every pair         z mean -0.20 (standard error 0.033), std 1.11, max 3.92; above 4: 0; above 2: 2.6% (normal: 2.3%)
unique pairs only  z mean -0.16 (standard error 0.029), std 0.99, max 3.03; above 4: 0; above 2: 1.5% (normal: 2.3%)
```

## `other_keys.py`

The same windows, distinct pairs only, scored with five other keys: is the mean below zero a property of one key? Takes its code from `human.py`.

In [7]:
!{sys.executable} other_keys.py

key  15485863: mean z -0.16, std 0.99, above 4: 0


key         1: mean z -0.03, std 0.97, above 4: 0


key         2: mean z -0.05, std 0.98, above 4: 0


key         3: mean z -0.24, std 0.97, above 4: 0


key         4: mean z  0.29, std 0.94, above 4: 0


key         5: mean z  0.02, std 1.02, above 4: 0


The post printed:

```text
key  15485863: mean z -0.16, std 0.99, above 4: 0
key         1: mean z -0.03, std 0.97, above 4: 0
key         2: mean z -0.05, std 0.98, above 4: 0
key         3: mean z -0.24, std 0.97, above 4: 0
key         4: mean z  0.29, std 0.94, above 4: 0
key         5: mean z  0.02, std 1.02, above 4: 0
```